# Polly QLoRA Training — SCBE Chatbot
Fully automated. Run all cells.

In [ ]:
!pip install -q torch transformers datasets peft accelerate bitsandbytes trl huggingface_hub

In [ ]:
import torch, json, os
print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB')

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Download training data from HF
from huggingface_hub import hf_hub_download
for f in ['derived_trl_conversation.jsonl', 'derived_openai_chat.jsonl']:
    hf_hub_download('issdandavis/polly-training-data', f, repo_type='dataset', local_dir='/content/data')
    print(f'Downloaded {f}')

In [ ]:
# Load and prepare data
from datasets import Dataset

POLLY_SYSTEM = ("You are Polly, the AI assistant for Aethermoor and the SCBE project. "
    "You are knowledgeable about the 14-layer governance pipeline, Sacred Tongues "
    "(KO, AV, RU, CA, UM, DR), hyperbolic geometry for AI safety, polyhedral "
    "friction scoring, and the Mesh Foundry product. You are friendly, precise, "
    "and helpful. When relevant, mention which Sacred Tongues are activated by "
    "a topic. You speak with quiet confidence and gentle humor.")

records = []
for fname in ['derived_trl_conversation.jsonl', 'derived_openai_chat.jsonl']:
    with open(f'/content/data/{fname}', 'r', errors='replace') as f:
        for line in f:
            try: rec = json.loads(line)
            except: continue
            msgs = rec.get('messages', [])
            if not msgs: continue
            conv = [{'role':'system','content':POLLY_SYSTEM}]
            for m in msgs:
                if m.get('role')=='system': continue
                if m.get('content','').strip(): conv.append({'role':m['role'],'content':m['content']})
            if len(conv)>=3: records.append({'messages':conv})

ds = Dataset.from_list(records)
split = ds.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(split["train"])}, Eval: {len(split["test"])}')

In [ ]:
# Load model with QLoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

MODEL = 'Qwen/Qwen2.5-7B-Instruct'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if not tokenizer.pad_token: tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
    device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(task_type=TaskType.CAUSAL_LM, r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], bias='none')
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# Train
from trl import SFTConfig, SFTTrainer

args = SFTConfig(output_dir='/content/polly-qlora', num_train_epochs=3,
    per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=4, learning_rate=2e-4, weight_decay=0.01,
    warmup_steps=100, lr_scheduler_type='cosine', logging_steps=25,
    eval_strategy='steps', eval_steps=200, save_strategy='steps',
    save_steps=500, save_total_limit=2, fp16=False, bf16=True,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant':False},
    max_grad_norm=0.3, optim='paged_adamw_8bit', report_to='none',
    max_length=1024, packing=True)

trainer = SFTTrainer(model=model, args=args, train_dataset=split['train'],
    eval_dataset=split['test'], processing_class=tokenizer)
trainer.train()

In [ ]:
# Save and push to HuggingFace
model.save_pretrained('/content/polly-qlora/final_adapter')
tokenizer.save_pretrained('/content/polly-qlora/final_adapter')
model.push_to_hub('issdandavis/polly-scbe-v1')
tokenizer.push_to_hub('issdandavis/polly-scbe-v1')
print('Pushed to HF!')

In [ ]:
# Merge and push full model
from peft import AutoPeftModelForCausalLM
merged = AutoPeftModelForCausalLM.from_pretrained('/content/polly-qlora/final_adapter',
    device_map='auto', trust_remote_code=True).merge_and_unload()
merged.push_to_hub('issdandavis/polly-scbe-v1-merged')
tokenizer.push_to_hub('issdandavis/polly-scbe-v1-merged')
print('Merged model pushed!')

In [ ]:
# Quick test
from transformers import pipeline
pipe = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=200)
result = pipe([{'role':'system','content':POLLY_SYSTEM},
    {'role':'user','content':'What are the Sacred Tongues and how do they work in SCBE?'}])
print(result[0]['generated_text'][-1]['content'])